# 03 - Silver Layer: Data Cleaning and Conformance

## Project
Ubuntu Retail Group - Legacy System to Modern Analytics Platform Migration

## Objective
Clean, standardise and conform the Bronze datasets to the System B target requirements.

## Activities
1. Load the Bronze Delta tables.
2. Identify records requiring correction or quarantine.
3. Standardise data types and formats.
4. Resolve duplicate primary keys.
5. Standardise categorical values and flags.
6. Validate relationships between datasets.
7. Write cleaned Silver Delta tables.
8. Maintain a bad_records quarantine table.

## Engineering Principles
- Never modify the Bronze source tables.
- Never silently discard records.
- Document all transformation and rejection decisions.
- Preserve original records for traceability.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from pyspark.sql.window import Window

In [0]:
bronze_customers = spark.table(
    "workspace.default.bronze_customers"
)

bronze_products = spark.table(
    "workspace.default.bronze_products"
)

bronze_orders = spark.table(
    "workspace.default.bronze_orders"
)

bronze_transactions = spark.table(
    "workspace.default.bronze_transactions"
)

datasets = {
    "customers": bronze_customers,
    "products": bronze_products,
    "orders": bronze_orders,
    "transactions": bronze_transactions
}

for name, df in datasets.items():
    print(f"{name}: {df.count():,} records")

In [0]:
bad_records_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("reason_code", StringType(), True),
    StructField("rejection_reason", StringType(), True),
    StructField("rejected_at", TimestampType(), True),
    StructField("raw_record", StringType(), True)
])

empty_bad_records = spark.createDataFrame(
    [],
    bad_records_schema
)

empty_bad_records.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bad_records")

In [0]:
for name, df in datasets.items():

    print(f"\n{'=' * 50}")
    print(f"{name.upper()} BRONZE DATA")
    print("=" * 50)

    df.printSchema()

    display(df.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_customers = spark.table(
    "workspace.default.bronze_customers"
)

print("Total customer records:", bronze_customers.count())

In [0]:
customer_duplicates = (
    bronze_customers
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("record_count")
    )
    .filter(F.col("record_count") > 1)
    .orderBy(F.desc("record_count"))
)

print(
    "Repeated customer IDs:",
    customer_duplicates.count()
)

display(customer_duplicates)

In [0]:
duplicate_customer_records = (
    bronze_customers
    .join(
        customer_duplicates.select("customer_id"),
        on="customer_id",
        how="inner"
    )
    .orderBy("customer_id")
)

display(duplicate_customer_records)

In [0]:
customer_completeness = (
    bronze_customers
    .withColumn(
        "completeness_score",
        F.when(
            F.col("full_name").isNotNull() &
            (F.trim("full_name") != ""),
            1
        ).otherwise(0)
        +
        F.when(
            F.col("email").isNotNull() &
            (F.trim("email") != ""),
            1
        ).otherwise(0)
        +
        F.when(
            F.col("phone").isNotNull() &
            (F.trim("phone") != ""),
            1
        ).otherwise(0)
        +
        F.when(F.col("registration_date").isNotNull(), 1).otherwise(0)
        +
        F.when(
            F.col("account_type").isNotNull() &
            (F.trim("account_type") != ""),
            1
        ).otherwise(0)
        +
        F.when(
            F.col("loyalty_tier").isNotNull() &
            (F.trim("loyalty_tier") != ""),
            1
        ).otherwise(0)
        +
        F.when(
            F.col("marketing_opt_in").isNotNull() &
            (F.trim("marketing_opt_in") != ""),
            1
        ).otherwise(0)
    )
)

duplicate_completeness = (
    customer_completeness
    .join(
        customer_duplicates.select("customer_id"),
        on="customer_id",
        how="inner"
    )
    .orderBy("customer_id", F.desc("completeness_score"))
)

display(duplicate_completeness)

In [0]:
customer_missing_analysis = bronze_customers.select(
    F.count(
        F.when(
            F.col("full_name").isNull() |
            (F.trim("full_name") == ""),
            1
        )
    ).alias("missing_names"),

    F.count(
        F.when(
            F.col("email").isNull() |
            (F.trim("email") == ""),
            1
        )
    ).alias("missing_emails"),

    F.count(
        F.when(
            F.col("loyalty_tier").isNull() |
            (F.trim("loyalty_tier") == ""),
            1
        )
    ).alias("missing_loyalty_tiers")
)

display(customer_missing_analysis)

In [0]:
display(
    bronze_customers.filter(
        F.col("customer_id").isNull() |
        (F.trim("customer_id") == "")
    )
)

In [0]:
customer_business_columns = [
    "customer_id",
    "full_name",
    "email",
    "phone",
    "registration_date",
    "account_type",
    "loyalty_tier",
    "marketing_opt_in"
]

customers_no_exact_duplicates = (
    bronze_customers
    .dropDuplicates(customer_business_columns)
)

print("Original records:", bronze_customers.count())
print("After exact deduplication:", customers_no_exact_duplicates.count())
print(
    "Exact duplicate rows removed:",
    bronze_customers.count() - customers_no_exact_duplicates.count()
)

In [0]:
comparison_columns = [
    "full_name",
    "email",
    "phone",
    "registration_date",
    "account_type",
    "loyalty_tier",
    "marketing_opt_in"
]

conflict_checks = [
    F.countDistinct(
        F.lower(F.trim(F.col(column).cast("string")))
    ).alias(f"{column}_variants")
    for column in comparison_columns
]

customer_conflict_summary = (
    customers_no_exact_duplicates
    .groupBy("customer_id")
    .agg(*conflict_checks)
)

conflict_condition = None

for column in comparison_columns:
    condition = F.col(f"{column}_variants") > 1

    conflict_condition = (
        condition if conflict_condition is None
        else conflict_condition | condition
    )

conflicting_customer_ids = (
    customer_conflict_summary
    .filter(conflict_condition)
    .select("customer_id")
)

print(
    "Customer IDs with conflicting information:",
    conflicting_customer_ids.count()
)

display(conflicting_customer_ids)

In [0]:
conflicting_customer_records = (
    customers_no_exact_duplicates
    .join(
        conflicting_customer_ids,
        on="customer_id",
        how="inner"
    )
)

resolvable_customer_records = (
    customers_no_exact_duplicates
    .join(
        conflicting_customer_ids,
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Records requiring review:",
    conflicting_customer_records.count()
)

print(
    "Records eligible for consolidation:",
    resolvable_customer_records.count()
)

In [0]:
def quarantine_records(df, source_file, reason_code, reason):

    quarantine_df = df.select(
        F.lit(source_file).alias("source_file"),
        F.lit(reason_code).alias("reason_code"),
        F.lit(reason).alias("rejection_reason"),
        F.current_timestamp().alias("rejected_at"),
        F.to_json(
            F.struct(*[
                F.col(column)
                for column in df.columns
            ])
        ).alias("raw_record")
    )

    quarantine_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("workspace.default.bad_records")

    print(f"Quarantined {df.count():,} records: {reason_code}")

In [0]:
quarantine_records(
    conflicting_customer_records,
    "customers_legacy_large.csv",
    "CUSTOMER_ID_CONFLICT",
    "Multiple records share a customer ID but contain conflicting non-null customer information."
)

In [0]:
aggregation_expressions = [
    F.first(F.col(column), ignorenulls=True).alias(column)
    for column in customer_business_columns
]

silver_customers_deduplicated = (
    resolvable_customer_records
    .groupBy("customer_id")
    .agg(*aggregation_expressions)
)

print(
    "Customer records after duplicate resolution:",
    silver_customers_deduplicated.count()
)

display(silver_customers_deduplicated)

In [0]:
column_names = silver_customers_deduplicated.columns
column_names[1] = "customer_id_duplicate"

silver_customers_checked = silver_customers_deduplicated.toDF(*column_names)

remaining_duplicates = (
    silver_customers_checked
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Remaining duplicate customer IDs:",
    remaining_duplicates.count()
)

display(remaining_duplicates)

In [0]:
%sql
SELECT
    reason_code,
    COUNT(*) AS quarantined_records
FROM workspace.default.bad_records
WHERE source_file = 'customers_legacy_large.csv'
GROUP BY reason_code;

In [0]:
customers_missing_names = (
    silver_customers_deduplicated
    .filter(
        F.col("full_name").isNull() |
        (F.trim(F.col("full_name")) == "")
    )
)

customers_with_names = (
    silver_customers_deduplicated
    .filter(
        F.col("full_name").isNotNull() &
        (F.trim(F.col("full_name")) != "")
    )
)

print("Customers with missing names:", customers_missing_names.count())
print("Customers with names:", customers_with_names.count())

In [0]:
missing_names_columns = customers_missing_names.columns
missing_names_columns[1] = "customer_id_duplicate"

customers_missing_names_clean = (
    customers_missing_names
    .toDF(*missing_names_columns)
    .drop("customer_id_duplicate")
)

quarantine_records(
    customers_missing_names_clean,
    "customers_legacy_large.csv",
    "MISSING_CUSTOMER_NAME",
    "Customer full_name is missing and cannot be split into target name fields."
)

In [0]:
customers_standardised = (
    customers_with_names
    .withColumn(
        "clean_full_name",
        F.trim(F.regexp_replace(F.col("full_name"), r"\s+", " "))
    )
    .withColumn(
        "name_parts",
        F.split(F.col("clean_full_name"), " ", 2)
    )
    .withColumn(
        "first_name",
        F.element_at(F.col("name_parts"), 1)
    )
    .withColumn(
        "last_name",
        F.when(
            F.size(F.col("name_parts")) > 1,
            F.element_at(F.col("name_parts"), 2)
        ).otherwise(F.lit(None).cast("string"))
    )
    .drop("full_name", "clean_full_name", "name_parts")
)

display(customers_standardised.limit(10))

In [0]:
standardised_columns = customers_standardised.columns
standardised_columns[1] = "customer_id_duplicate"

customers_standardised = (
    customers_standardised
    .toDF(*standardised_columns)
    .drop("customer_id_duplicate")
)

customers_standardised = (
    customers_standardised
    .withColumn(
        "email",
        F.when(
            F.trim(F.col("email")) == "",
            None
        ).otherwise(
            F.lower(F.trim(F.col("email")))
        )
    )
)

display(
    customers_standardised
    .select("customer_id", "email")
    .limit(20)
)

In [0]:
customers_standardised = (
    customers_standardised
    .withColumn(
        "account_type",
        F.when(
            F.trim(F.col("account_type")) == "",
            None
        ).otherwise(
            F.initcap(F.lower(F.trim(F.col("account_type"))))
        )
    )
    .withColumn(
        "loyalty_tier",
        F.when(
            F.trim(F.col("loyalty_tier")) == "",
            None
        ).otherwise(
            F.initcap(F.lower(F.trim(F.col("loyalty_tier"))))
        )
    )
)

display(
    customers_standardised
    .groupBy("account_type")
    .count()
    .orderBy("account_type")
)

display(
    customers_standardised
    .groupBy("loyalty_tier")
    .count()
    .orderBy("loyalty_tier")
)

In [0]:
marketing_flag = F.lower(
    F.trim(F.col("marketing_opt_in").cast("string"))
)

customers_standardised = (
    customers_standardised
    .withColumn(
        "marketing_opt_in",
        F.when(
            marketing_flag.isin("true", "1", "y", "yes"),
            F.lit(True)
        )
        .when(
            marketing_flag.isin("false", "0", "n", "no"),
            F.lit(False)
        )
        .otherwise(F.lit(None).cast("boolean"))
    )
)

In [0]:
display(
    customers_standardised
    .groupBy("marketing_opt_in")
    .count()
)

In [0]:
customers_standardised = (
    customers_standardised
    .withColumn(
        "phone",
        F.when(
            F.trim(F.col("phone")) == "",
            None
        ).otherwise(F.trim(F.col("phone")))
    )
)

display(
    customers_standardised
    .select("customer_id", "phone")
    .limit(20)
)

In [0]:
display(
    customers_standardised
    .select("registration_date")
    .distinct()
    .orderBy("registration_date")
)

In [0]:
display(customers_standardised.limit(20))

customers_standardised.printSchema()

In [0]:
display(
    customers_standardised
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
# Parse the supported registration-date formats
customers_with_parsed_dates = (
    customers_standardised
    .withColumn(
        "parsed_registration_date",
        F.coalesce(
            F.to_date(F.trim(F.col("registration_date")), "yyyy-MM-dd"),
            F.to_date(F.trim(F.col("registration_date")), "MM-dd-yyyy"),
            F.to_date(F.trim(F.col("registration_date")), "dd/MM/yyyy")
        )
    )
)

In [0]:
customers_with_parsed_dates = (
    customers_with_parsed_dates
    .withColumn(
        "parsed_registration_date",
        F.coalesce(
            F.try_to_date(F.trim(F.col("registration_date")), "yyyy-MM-dd"),
            F.try_to_date(F.trim(F.col("registration_date")), "MM-dd-yyyy"),
            F.try_to_date(F.trim(F.col("registration_date")), "dd/MM/yyyy")
        )
    )
)

invalid_registration_dates = (
    customers_with_parsed_dates
    .filter(
        F.col("registration_date").isNotNull() &
        (F.trim(F.col("registration_date")) != "") &
        F.col("parsed_registration_date").isNull()
    )
)

valid_registration_dates = (
    customers_with_parsed_dates
    .filter(
        F.col("registration_date").isNull() |
        (F.trim(F.col("registration_date")) == "") |
        F.col("parsed_registration_date").isNotNull()
    )
)

print("Invalid registration dates:", invalid_registration_dates.count())
print("Records retained:", valid_registration_dates.count())

In [0]:
quarantine_records(
    invalid_registration_dates,
    "customers_legacy_large.csv",
    "INVALID_REGISTRATION_DATE",
    "Registration date does not match any of the documented supported date formats."
)

In [0]:
silver_customers_final = (
    valid_registration_dates
    .drop("registration_date")
    .withColumnRenamed(
        "parsed_registration_date",
        "registration_date"
    )
)

In [0]:
silver_customers_final.printSchema()

In [0]:
duplicate_customer_ids = (
    silver_customers_final
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:", duplicate_customer_ids.count())

display(duplicate_customer_ids)

In [0]:
customer_key_issues = silver_customers_final.filter(
    F.col("customer_id").isNull() |
    (F.trim(F.col("customer_id")) == "") |
    F.col("first_name").isNull() |
    (F.trim(F.col("first_name")) == "")
)

print("Records with mandatory-field issues:", customer_key_issues.count())

In [0]:
display(silver_customers_final.limit(20))
silver_customers_final.printSchema()

In [0]:
print("Final Silver customer records:", silver_customers_final.count())
print("Quarantined customer records:")

display(
    spark.table("workspace.default.bad_records")
    .filter(
        F.col("source_file") == "customers_legacy_large.csv"
    )
    .groupBy("reason_code")
    .count()
)

In [0]:
(
    silver_customers_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_customers")
)

print("silver_customers created successfully!")

In [0]:
saved_customers = spark.table(
    "workspace.default.silver_customers"
)

print("Saved records:", saved_customers.count())

saved_customers.printSchema()

display(saved_customers.limit(10))

## Customer Silver Transformation Summary

The customer dataset was processed from the Bronze layer into the Silver layer.

### Transformations performed
- Removed exact duplicate business records.
- Investigated repeated customer IDs.
- Consolidated non-conflicting records.
- Quarantined records with conflicting customer information.
- Quarantined records with missing customer names.
- Split full_name into first_name and last_name.
- Standardised email addresses to lowercase.
- Standardised account_type and loyalty_tier casing.
- Converted marketing_opt_in into BooleanType.
- Trimmed phone numbers while preserving them as strings.
- Converted registration_date into DateType.

### Data Quality
- Primary-key uniqueness was checked.
- Mandatory customer fields were validated.
- Invalid registration dates were quarantined.
- Original Bronze records were preserved.

### Target Table
workspace.default.silver_customers